In [1]:
"""
═══════════════════════════════════════════════════════════════════════════
NETWORK DATA — FINAL PIPELINE (v4 corrected, local-only)
All files local. STRING files NOT compressed. No URL fallback mechanism.
Hierarchy exclusion: root-depth-0 AND size > ROOT_EXCLUDE_MIN_SIZE.
HGNC Pass 2 column detection fixed to find "NCBI Gene ID".
═══════════════════════════════════════════════════════════════════════════
"""

import os, json, csv, io
from datetime import datetime, timezone
from collections import defaultdict, deque
import numpy as np

# ─────────────────────────────────────────────────────────────────────────
# ALL PATHS — LOCAL ONLY, NO FALLBACKS
# ─────────────────────────────────────────────────────────────────────────
LANDMARK_GENES_PATH  = "/kaggle/input/datasets/apexblue/lincs-new/pathway_landmark_genes.txt"
LINCS_GENE_INFO_PATH = "/kaggle/input/datasets/apexblue/lincs-new/GSE92742_Broad_LINCS_gene_info.txt"
GMT_LOCAL            = "/kaggle/input/datasets/apexblue/lincs-new/ReactomePathways.gmt"
RELATION_LOCAL       = ""   # set to local path if you have it, else provide URL below
RELATION_URL         = "https://download.reactome.org/96/ReactomePathwaysRelation.txt"   # e.g. https://reactome.org/download/current/ReactomePathwaysRelation.txt
STRING_INFO_LOCAL    = "/kaggle/input/datasets/apexblue/lincs-new/9606.protein.info.v12.0.txt"
STRING_LINKS_LOCAL   = "/kaggle/input/datasets/apexblue/lincs-new/9606.protein.links.v12.0.txt"
HGNC_LOCAL           = ""   # set to local path if you have it, else URL below is used
HGNC_URL             = "https://www.genenames.org/cgi-bin/download/custom?col=gd_hgnc_id&col=gd_app_sym&col=gd_app_name&col=gd_status&col=gd_prev_sym&col=gd_aliases&col=gd_pub_chrom_map&col=gd_pub_acc_ids&col=gd_pub_refseq_ids&col=gd_pub_eg_id&status=Approved&status=Entry%20Withdrawn&hgnc_dbtag=on&order_by=gd_app_sym_sort&format=text&submit=submit"
# NOTE: gd_pub_eg_id added to URL to include Entrez Gene ID column

OUT_DIR = "/kaggle/working/network_data_final/"
os.makedirs(OUT_DIR, exist_ok=True)

MIN_SIZE               = 10
ROOT_EXCLUDE_MIN_SIZE  = 70   # exclude root nodes ONLY if size > this
STRING_SCORE_THRESHOLD = 400
PERCENTILE_FOR_A_NORM  = 99

print(f"Config: MIN_SIZE={MIN_SIZE}, ROOT_EXCLUDE_MIN_SIZE={ROOT_EXCLUDE_MIN_SIZE}")

# ─────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────
def load_text_local_or_url(local_path, url, timeout=120):
    """Load text file — local first (no decompression), URL fallback."""
    if local_path and os.path.exists(local_path):
        print(f"  Local: {local_path}")
        with open(local_path, 'r', encoding='utf-8') as f:
            return f.read()
    if not url or url.startswith("FILL_IN"):
        raise FileNotFoundError(f"Local path not found and URL not set: {local_path}")
    print(f"  Downloading: {url}")
    import urllib.request
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read().decode('utf-8')

# ═════════════════════════════════════════════════════════════════════════
# STEP 0 — LANDMARK GENES + ENTREZ IDs
# ═════════════════════════════════════════════════════════════════════════
with open(LANDMARK_GENES_PATH) as f:
    landmark_genes = [l.strip() for l in f.readlines()]
assert len(landmark_genes) == 978, f"Expected 978 genes, got {len(landmark_genes)}"
landmark_set  = set(landmark_genes)
gene_to_idx   = {g: i for i, g in enumerate(landmark_genes)}

landmark_to_entrez = {}
with open(LINCS_GENE_INFO_PATH) as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        sym = row.get('pr_gene_symbol', '').strip().upper()
        if sym in landmark_set:
            landmark_to_entrez[sym] = str(row.get('pr_gene_id', '')).strip()

print(f"Landmark genes: 978  |  With Entrez IDs: {len(landmark_to_entrez)}")

# ═════════════════════════════════════════════════════════════════════════
# STEP 1 — GMT
# ═════════════════════════════════════════════════════════════════════════
print("\nSTEP 1 — Reactome GMT")
with open(GMT_LOCAL, 'r', encoding='utf-8') as f:
    gmt_lines = f.read().strip().split('\n')
print(f"  GMT lines: {len(gmt_lines)}")

pathway_genes, pathway_names = {}, {}
for line in gmt_lines:
    parts = line.split('\t')
    if len(parts) < 3:
        continue
    name, url_or_id, genes = parts[0], parts[1], parts[2:]
    pid = None
    for field in [name, url_or_id]:
        if 'R-HSA-' in field:
            pid = 'R-HSA-' + field.split('R-HSA-')[1].split()[0].split('/')[0]
            break
    if pid is None:
        continue
    pathway_genes[pid]  = {g.strip().upper() for g in genes}
    pathway_names[pid]  = name

print(f"  R-HSA pathways: {len(pathway_genes)}")
gmt_gene_universe = set().union(*pathway_genes.values())

# ═════════════════════════════════════════════════════════════════════════
# STEP 2 — PASS 0: DIRECT MATCH
# ═════════════════════════════════════════════════════════════════════════
covered_pass0 = set()
for genes in pathway_genes.values():
    covered_pass0 |= (genes & landmark_set)
still_missing = landmark_set - covered_pass0
print(f"\nPass 0 — {len(covered_pass0)}/978 covered  ({len(still_missing)} still missing)")

# ═════════════════════════════════════════════════════════════════════════
# STEP 3 — PASS 1: MYGENE (Entrez-verified)
# ═════════════════════════════════════════════════════════════════════════
print("\nSTEP 3 — Pass 1: mygene alias resolution")
try:
    import mygene
except ImportError:
    os.system("pip install mygene --quiet"); import mygene
mg = mygene.MyGeneInfo()

candidates_to_check = {g: landmark_to_entrez[g]
                       for g in still_missing if g in landmark_to_entrez}
entrez_list = list(candidates_to_check.values())
print(f"  Querying mygene for {len(entrez_list)} missing genes (alias+other_names+symbol)...")

alias_results = mg.querymany(entrez_list, scopes='entrezgene',
                              fields='alias,other_names,symbol',
                              species='human', verbose=False)
entrez_to_all_names = {}
for r in alias_results:
    if r.get('notfound'):
        continue
    names = set()
    for field in ['alias', 'other_names']:
        val = r.get(field, [])
        if isinstance(val, str): val = [val]
        names.update(v.strip().upper() for v in val)
    entrez_to_all_names[r['query']] = names

raw_candidates = []
for sym, eid in candidates_to_check.items():
    for cand in entrez_to_all_names.get(eid, set()):
        if cand in gmt_gene_universe and cand not in landmark_set:
            raw_candidates.append((sym, eid, cand))

unique_aliases = list({c[2] for c in raw_candidates})
print(f"  {len(raw_candidates)} raw candidates, {len(unique_aliases)} unique — verifying...")

verify_results = mg.querymany(unique_aliases, scopes='symbol,alias',
                               fields='entrezgene',
                               species='human', verbose=False)
alias_to_verified = {}
for r in verify_results:
    if not r.get('notfound') and r.get('entrezgene') is not None:
        alias_to_verified[r['query']] = str(r['entrezgene'])

pass1_accepted, pass1_rejected = {}, []
for sym, eid, alias in raw_candidates:
    if alias in pass1_accepted: continue
    if alias_to_verified.get(alias) == eid:
        pass1_accepted[alias] = sym
    else:
        pass1_rejected.append({"gmt_symbol": alias, "claimed_landmark": sym,
                                "claimed_entrez": eid,
                                "actual_entrez": alias_to_verified.get(alias)})
print(f"  Pass 1 accepted: {len(pass1_accepted)}, rejected: {len(pass1_rejected)}")
for r in pass1_rejected:
    print(f"    REJECT: GMT '{r['gmt_symbol']}' → Entrez {r['actual_entrez']} "
          f"≠ '{r['claimed_landmark']}' Entrez {r['claimed_entrez']}")

# ═════════════════════════════════════════════════════════════════════════
# STEP 4 — PASS 2: HGNC (collision-checked)
# ═════════════════════════════════════════════════════════════════════════
print("\nSTEP 4 — Pass 2: HGNC alias resolution")
still_missing_p2 = {g for g in still_missing if g not in pass1_accepted.values()}

hgnc_text = load_text_local_or_url(HGNC_LOCAL, HGNC_URL)
hgnc_text  = hgnc_text.lstrip('\ufeff')
hgnc_reader = csv.DictReader(io.StringIO(hgnc_text), delimiter='\t')
raw_headers = hgnc_reader.fieldnames or []
print(f"  HGNC headers (first 5): {raw_headers[:5]}")
norm_h = {h.strip().lower(): h for h in raw_headers}

def find_col(norm_map, candidates):
    for c in candidates:
        cl = c.lower()
        if cl in norm_map:
            return norm_map[cl]
    return None

# FIX: "NCBI Gene ID" is the actual Entrez column name in your HGNC file
col_entrez = find_col(norm_h, [
    'ncbi gene id',                    # ← your file uses this
    'ncbi gene id(supplied by ncbi)',   # alternate form
    'entrez gene id',
    'entrez gene id(supplied by ncbi)',
    'entrez_id',
    'gd_pub_eg_id',
])
col_symbol = find_col(norm_h, ['approved symbol', 'symbol'])
col_alias  = find_col(norm_h, ['alias symbols', 'alias_symbol'])
col_prev   = find_col(norm_h, ['previous symbols', 'prev_symbol'])

print(f"  Detected — entrez: '{col_entrez}', alias: '{col_alias}', prev: '{col_prev}'")
if col_entrez is None:
    print("  WARNING: Entrez column not found in HGNC file. Pass 2 skipped.")

hgnc_by_entrez, hgnc_symbol_to_entrez = {}, {}
if col_entrez:
    for rec in hgnc_reader:
        eid = (rec.get(col_entrez) or '').strip()
        sym = (rec.get(col_symbol) or '').strip().upper()
        if eid: hgnc_by_entrez[eid] = rec
        if sym: hgnc_symbol_to_entrez[sym] = eid
    print(f"  HGNC records loaded: {len(hgnc_by_entrez)}")

pass2_accepted, pass2_rejected = {}, []
if col_entrez:
    for sym in still_missing_p2:
        eid = landmark_to_entrez.get(sym)
        rec = hgnc_by_entrez.get(eid) if eid else None
        if not rec: continue
        cand_set = set()
        for col in [col_alias, col_prev]:
            if col:
                field = (rec.get(col) or '').replace('"', '').replace('|', ' ')
                for tok in field.split():
                    tok = tok.strip().upper().rstrip(',')
                    if tok: cand_set.add(tok)
        for cand in cand_set:
            if cand in gmt_gene_universe and cand not in landmark_set and cand not in pass2_accepted:
                owner_eid = hgnc_symbol_to_entrez.get(cand)
                if owner_eid and owner_eid != eid:
                    pass2_rejected.append({"gmt_symbol": cand, "claimed_landmark": sym,
                                           "collision_with_entrez": owner_eid})
                else:
                    pass2_accepted[cand] = sym
                    break

print(f"  Pass 2 accepted: {len(pass2_accepted)}, rejected: {len(pass2_rejected)}")

ALIAS_MAP = {**pass1_accepted, **pass2_accepted}
print(f"\nTotal aliases resolved: {len(ALIAS_MAP)}")
for gmt_sym, lincs_sym in sorted(ALIAS_MAP.items()):
    print(f"  GMT '{gmt_sym}' → LINCS '{lincs_sym}'")

# ═════════════════════════════════════════════════════════════════════════
# STEP 5 — RESCAN WITH ALIASES
# ═════════════════════════════════════════════════════════════════════════
pathway_landmark_genes = {}
for pid, genes in pathway_genes.items():
    found = set()
    for g in genes:
        if g in landmark_set:       found.add(g)
        elif g in ALIAS_MAP:        found.add(ALIAS_MAP[g])
    pathway_landmark_genes[pid] = found
pathway_sizes_all = {pid: len(g) for pid, g in pathway_landmark_genes.items()}

# ═════════════════════════════════════════════════════════════════════════
# STEP 6 — HIERARCHY: ROOTS + DEPTH
# ═════════════════════════════════════════════════════════════════════════
print("\nSTEP 6 — Pathway hierarchy")
relation_text = load_text_local_or_url(RELATION_LOCAL, RELATION_URL)
parent_to_children, child_set = defaultdict(list), set()
n_non_human = 0
for line in relation_text.strip().split('\n'):
    parts = line.strip().split('\t')
    if len(parts) != 2: continue
    p, c = parts
    if p.startswith('R-HSA-') and c.startswith('R-HSA-'):
        parent_to_children[p].append(c)
        child_set.add(c)
    else:
        n_non_human += 1

root_pathways = {pid for pid in pathway_genes if pid not in child_set}
print(f"  R-HSA root pathways: {len(root_pathways)}  (non-human edges filtered: {n_non_human})")

depth = {pid: 0 for pid in root_pathways}
queue = deque(root_pathways)
while queue:
    cur = queue.popleft()
    for child in parent_to_children.get(cur, []):
        nd = depth[cur] + 1
        if child not in depth or nd < depth[child]:
            depth[child] = nd
            queue.append(child)

# ═════════════════════════════════════════════════════════════════════════
# STEP 7 — APPLY FILTERS
# ═════════════════════════════════════════════════════════════════════════
def is_excluded(pid):
    size = pathway_sizes_all[pid]
    if size < MIN_SIZE:
        return True, "below_min_size"
    if depth.get(pid, 99) == 0 and size > ROOT_EXCLUDE_MIN_SIZE:
        return True, "root_umbrella"
    return False, None

final_pathway_ids = sorted([pid for pid in pathway_sizes_all
                             if not is_excluded(pid)[0]])
N_p = len(final_pathway_ids)

excluded_roots = sorted(
    [(pid, pathway_names.get(pid,'?'), pathway_sizes_all[pid])
     for pid in pathway_sizes_all
     if is_excluded(pid)[1] == "root_umbrella"],
    key=lambda x: -x[2])

print(f"\n{'='*70}\nSTEP 7 — FINAL PATHWAY SET\n{'='*70}")
print(f"Candidates with size >= {MIN_SIZE}: "
      f"{sum(1 for _,r in [(p,is_excluded(p)) for p in pathway_sizes_all] if not r[0] or r[1]=='root_umbrella')}")
print(f"Excluded as large root umbrella (depth=0, size>{ROOT_EXCLUDE_MIN_SIZE}): {len(excluded_roots)}")
for pid, name, size in excluded_roots:
    print(f"   {size:4d} genes — {name}  ({pid})")
print(f"\nFINAL N_p: {N_p}")

# ═════════════════════════════════════════════════════════════════════════
# STEP 8 — BUILD M, M_NORM
# ═════════════════════════════════════════════════════════════════════════
M = np.zeros((N_p, 978), dtype=np.int8)
for i, pid in enumerate(final_pathway_ids):
    for g in pathway_landmark_genes[pid]:
        M[i, gene_to_idx[g]] = 1
M_norm = M.astype(np.float32) / M.sum(axis=1, keepdims=True).astype(np.float32)
final_coverage = M.sum(axis=0) > 0
final_missing  = [landmark_genes[i] for i in range(978) if not final_coverage[i]]
print(f"\nM: {M.shape}  |  Coverage: {final_coverage.sum()}/978  "
      f"(missing: {len(final_missing)})")

# ═════════════════════════════════════════════════════════════════════════
# STEP 9 — BUILD A (percentile-normalised, rebuilt from filtered M)
# ═════════════════════════════════════════════════════════════════════════
A_raw = M.astype(np.float32).T @ M.astype(np.float32)
np.fill_diagonal(A_raw, 0)
iu = np.triu_indices(978, k=1)
raw_nz = A_raw[iu]; raw_nz = raw_nz[raw_nz > 0]
norm_ref = float(np.percentile(raw_nz, PERCENTILE_FOR_A_NORM))
A = np.clip(A_raw / norm_ref, 0, 1.0).astype(np.float32)
assert np.allclose(A, A.T, atol=1e-6) and np.diag(A).sum() == 0
a_nz = A[iu]; a_nz = a_nz[a_nz > 0]
print(f"\nA: nonzero fraction={a_nz.size/len(iu[0]):.4f}  "
      f"p99 norm ref={norm_ref:.2f}  frac<0.1: {(a_nz<0.1).mean():.1%}")

# ═════════════════════════════════════════════════════════════════════════
# STEP 10 — STRING (local, NOT compressed)
# ═════════════════════════════════════════════════════════════════════════
print(f"\nSTEP 10 — STRING (local, uncompressed)")
protein_to_symbol = {}
with open(STRING_INFO_LOCAL, 'r', encoding='utf-8') as f:
    next(f)   # skip header
    for line in f:
        parts = line.split('\t')
        if len(parts) >= 2:
            protein_to_symbol[parts[0].strip()] = parts[1].strip()
print(f"  Proteins loaded: {len(protein_to_symbol)}")

STRING_adj = np.zeros((978, 978), dtype=np.float32)
n_used = 0
with open(STRING_LINKS_LOCAL, 'r', encoding='utf-8') as f:
    next(f)   # skip header
    for line in f:
        parts = line.split()
        if len(parts) != 3: continue
        p1, p2, score = parts
        score = int(score)
        if score < STRING_SCORE_THRESHOLD: continue
        s1, s2 = protein_to_symbol.get(p1), protein_to_symbol.get(p2)
        if s1 in gene_to_idx and s2 in gene_to_idx:
            i, j = gene_to_idx[s1], gene_to_idx[s2]
            v = score / 1000.0
            STRING_adj[i, j] = max(STRING_adj[i, j], v)
            STRING_adj[j, i] = max(STRING_adj[j, i], v)
            n_used += 1
np.fill_diagonal(STRING_adj, 0)
print(f"  Edges used: {n_used}  |  Nonzero fraction: {(STRING_adj>0).mean():.4f}")

# ═════════════════════════════════════════════════════════════════════════
# STEP 11 — SAVE
# ═════════════════════════════════════════════════════════════════════════
np.save(OUT_DIR + "M_reactome.npy",      M)
np.save(OUT_DIR + "M_norm_reactome.npy", M_norm)
np.save(OUT_DIR + "A_copathway.npy",     A)
np.save(OUT_DIR + "STRING_adj_978.npy",  STRING_adj)
with open(OUT_DIR + "pathway_landmark_genes.txt", "w") as f:
    f.write('\n'.join(landmark_genes))
with open(OUT_DIR + "pathway_info.tsv", "w", newline='') as f:
    w = csv.writer(f, delimiter='\t')
    w.writerow(['pathway_id','pathway_name','n_landmark_genes','landmark_gene_symbols'])
    for pid in final_pathway_ids:
        w.writerow([pid, pathway_names.get(pid,'?'),
                    len(pathway_landmark_genes[pid]),
                    ','.join(sorted(pathway_landmark_genes[pid]))])

provenance = {
    "pipeline_version": "v4_corrected_local",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "config": {"min_size": MIN_SIZE, "root_exclude_min_size": ROOT_EXCLUDE_MIN_SIZE,
               "hierarchy_logic": "root-depth-0 AND n_lm_genes > ROOT_EXCLUDE_MIN_SIZE",
               "percentile_for_a_norm": PERCENTILE_FOR_A_NORM,
               "string_score_threshold": STRING_SCORE_THRESHOLD},
    "alias_resolution": {
        "pass0_direct": len(covered_pass0),
        "pass1_mygene": {"accepted": pass1_accepted, "rejected": pass1_rejected},
        "pass2_hgnc":   {"detected_entrez_col": col_entrez,
                         "accepted": pass2_accepted, "rejected": pass2_rejected},
        "total_resolved": len(ALIAS_MAP), "complete_map": ALIAS_MAP,
    },
    "reactome": {
        "total_rhsa_in_gmt": len(pathway_genes),
        "candidates_after_min_size":
            sum(1 for s in pathway_sizes_all.values() if s >= MIN_SIZE),
        "excluded_as_root_umbrella": len(excluded_roots),
        "excluded_root_pathways": [{"id":p,"name":n,"size":s} for p,n,s in excluded_roots],
        "final_N_p": N_p,
        "landmark_genes_covered": int(final_coverage.sum()),
        "landmark_genes_missing": final_missing,
    },
    "A_matrix": {"normalisation": f"p{PERCENTILE_FOR_A_NORM}_capped",
                 "norm_ref": norm_ref,
                 "nonzero_fraction": float(a_nz.size/len(iu[0])),
                 "frac_below_0.1": float((a_nz<0.1).mean())},
    "string": {"edges_used": n_used, "nonzero_fraction": float((STRING_adj>0).mean()),
               "note": "NOT used in V2 training"},
    "canonical_gene_order": {"source": LANDMARK_GENES_PATH, "n_genes": 978,
                              "first": landmark_genes[0], "last": landmark_genes[-1]},
}
with open(OUT_DIR + "network_data_provenance.json", "w") as f:
    json.dump(provenance, f, indent=2)

# ═════════════════════════════════════════════════════════════════════════
# STEP 12 — VALIDATION
# ═════════════════════════════════════════════════════════════════════════
print(f"\n{'='*70}\nSTEP 12 — VALIDATION\n{'='*70}")
assert M.shape == (N_p, 978) and M_norm.shape == M.shape
assert A.shape == (978, 978) and STRING_adj.shape == (978, 978)
assert M.dtype == np.int8 and M_norm.dtype == np.float32 and A.dtype == np.float32
assert set(np.unique(M)).issubset({0, 1})
assert np.abs(M_norm.sum(axis=1) - 1.0).max() < 1e-4
assert A.min() >= 0 and A.max() <= 1.0
assert np.allclose(A, A.T, atol=1e-6) and np.diag(A).sum() == 0
assert np.allclose(STRING_adj, STRING_adj.T, atol=1e-6) and np.diag(STRING_adj).sum() == 0
assert not any(np.isnan(x).any() for x in [M_norm, A, STRING_adj])
print("ALL ASSERTIONS PASSED")

print(f"\n{'='*70}\nFINAL SUMMARY\n{'='*70}")
print(f"N_p: {N_p}  |  Coverage: {final_coverage.sum()}/978  "
      f"|  Aliases resolved: {len(ALIAS_MAP)}")
print(f"Roots excluded: {len(excluded_roots)} (all depth=0, size>{ROOT_EXCLUDE_MIN_SIZE})")
print(f"A: nonzero={a_nz.size/len(iu[0]):.4f}, frac<0.1={(a_nz<0.1).mean():.1%}")
print(f"Saved to: {OUT_DIR}")

Config: MIN_SIZE=10, ROOT_EXCLUDE_MIN_SIZE=70
Landmark genes: 978  |  With Entrez IDs: 978

STEP 1 — Reactome GMT
  GMT lines: 2855
  R-HSA pathways: 2855

Pass 0 — 747/978 covered  (231 still missing)

STEP 3 — Pass 1: mygene alias resolution
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
  Querying mygene for 231 missing genes (alias+other_names+symbol)...
  10 raw candidates, 10 unique — verifying...
  Pass 1 accepted: 1, rejected: 9
    REJECT: GMT 'CHD5' → Entrez 26038 ≠ 'WRB' Entrez 7485
    REJECT: GMT 'PRF1' → Entrez 5551 ≠ 'ZNF395' Entrez 55893
    REJECT: GMT 'RRP1' → Entrez 8568 ≠ 'RRP1B' Entrez 23076
    REJECT: GMT 'POLK' → Entrez 51426 ≠ 'PAPD7' Entrez 11044
    REJECT: GMT 'GOT1' → Entrez 2805 ≠ 'GOLT1B' Entrez 51026
    REJECT: GMT 'OSR1' → Entrez 130497 ≠ 'OXSR1' Entrez 9943
    REJECT: GMT 'HK2' → Entrez 3099 ≠ 'HOOK2' Entrez 29911
    REJECT: GMT 'DMP1' → Entrez 1758 ≠ 'DMTF1' Entrez 9988
    REJECT: GMT 'ARC' → Entrez 23237 ≠ 'NOL3' En